In [1]:
import warnings
warnings.filterwarnings('ignore')

(Cookbook_Saving_ionic_interactions)=
# Saving ionic analyses
*Calculating, naming, persisting and querying formal-charge contacts.*

Store the calculated analysis with its molecular system in H5MSM 0.5, then
recover it through public conversion. The example uses explicit isolated-ion
chemistry and constructed coordinates; it makes no force-field or energy claim.

:::{versionadded} 1.0.0
:::

## Preparing the system

The first structure has a contact; the second has none. Declare the completeness
assumption explicitly for these isolated ions.

In [2]:
import numpy as np
import molsysmt as msm
from molsysmt import pyunitwizard as puw
from molsysmt.native import MolSys, Topology

molsys = MolSys()
molsys.topology = Topology(n_atoms=2)
molsys.topology.atoms['atom_type'] = ['Na', 'Cl']
msm.set(molsys, element='atom', formal_charge=[1, -1])
molsys.structures.append(coordinates=puw.quantity(np.array([
    [[0,0,0], [0.3,0,0]], [[0,0,0], [2,0,0]],
]), 'nm'))

## Calculating and naming the analysis

Check the result before attaching it. Mapping assignment validates that its
atom and structure axes match the destination; it declares correspondence and
does not authenticate molecular origin.

In [3]:
analysis = msm.interactions.ionic.get_ionic_interactions(
    molsys, '0.4 nm', pbc=False, assume_complete_connectivity=True)
assert not molsys.interactions
molsys.interactions = {**molsys.interactions, 'ionic': analysis}
print(analysis.evaluated_structure_indices.tolist())
print(analysis.measure_units)
print(analysis.software)

[0, 1]
{'distance': 'nm', 'positive_charge': 'e', 'negative_charge': 'e'}
{'molsysmt': '0.21.0+606.ga03eb4bf6'}


## Saving and recovering

The temporary file below contains both coordinates and the named analysis.
H5MSM preserves the version that produced the observations, even if the version
used to load it later differs.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    filename = str(Path(directory) / 'ionic.h5msm')
    msm.convert(molsys, to_form=filename)
    restored = msm.convert(filename, to_form='molsysmt.MolSys')

saved = restored.interactions['ionic']
assert saved.software == analysis.software
assert saved.parameters == analysis.parameters
np.testing.assert_allclose(saved.measurements['distance'], [0.3])
print(list(restored.interactions))

['ionic']


## Querying a frame or atom

The empty frame is evaluated, while occurrence indices identify the observations
within this version of the analysis. Querying a saved result requires no new
scientific calculation.

In [5]:
visible = saved.query(structure_indices=[0], atom_indices=[0]).to_dict()
empty = saved.query(structure_indices=[1]).to_dict()
print(visible['occurrence_indices'].tolist())
print(empty['evaluated_structure_indices'].tolist())
assert empty['occurrence_indices'].size == 0
assert empty['evaluated_structure_indices'].tolist() == [1]

[0]
[1]


:::{seealso}
:class: dropdown

- {ref}`Tutorial_Get_ionic_interactions` — scope, PBC, state and coverage limits.
- {func}`molsysmt.basic.convert` — public result and H5MSM conversion.
- {ref}`user-tools-interactions-result` — sparse participant and frame queries.
:::